In [ ]:
import pandas as pd
import plotly.express as px
import mercury as mr
import warnings
warnings.filterwarnings('ignore')

# 1. Carga de datos
df_raw = pd.read_csv('datos/datos_educacion_pib_limpio.csv')

# 2. Limpieza Institucional (Usamos regex=False para evitar el bug de "Ciudad de Estado de México")
nombres_pro = {
    'Ciudad de MÃ©xico': 'Ciudad de México',
    'MÃ©xico': 'Estado de México', 
    'QuerÃ©taro': 'Querétaro', 
    'Nuevo LeÃ³n': 'Nuevo León',
    'MichoacÃ¡n de Ocampo': 'Michoacán', 
    'San Luis PotosÃ­': 'San Luis Potosí',
    'YucatÃ¡n': 'Yucatán', 
    'Coahuila de Zaragoza': 'Coahuila',
    'Veracruz de Ignacio de la Llave': 'Veracruz'
}
df_raw['Estado'] = df_raw['Estado'].replace(nombres_pro, regex=False)

# 3. Filtro de duplicados (Aseguramos las 32 entidades)
df = df_raw.drop_duplicates(subset=['Estado'], keep='first').copy()

<style>
    /* Estilo Institucional / Reporte Ejecutivo */
    body { background-color: #FFFFFF !important; }
    .jp-RenderedHTMLCommon {
        max-width: 1000px !important;
        margin: 0 auto !important;
        padding: 40px 20px !important;
        font-family: 'Arial', 'Helvetica', sans-serif !important;
        color: #333333;
        line-height: 1.6;
    }
    h1 { font-size: 2.2em !important; font-weight: bold; text-align: left; color: #002060; border-bottom: 2px solid #7B241C; padding-bottom: 10px; text-transform: uppercase; }
    
    /* Cajas de Resumen */
    .resumen { font-size: 1.1em; color: #444; border-left: 3px solid #002060; padding-left: 15px; margin-bottom: 30px; background-color: #FAFAFA; padding: 15px; }

    /* LOS NUEVOS SEPARADORES DE SECCIÓN (LEGO) */
    .section-break {
        background-color: #002060; /* Azul Marino Corporativo */
        padding: 30px 40px !important;
        margin: 60px 0 30px 0 !important; 
        border-left: 10px solid #7B241C; /* Acento Guinda */
        border-radius: 4px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
    }
    .section-break h2 {
        color: #FFFFFF !important;
        background-color: transparent !important;
        border: none !important;
        margin: 0 !important;
        padding: 0 !important;
        font-size: 1.6em !important;
        text-transform: uppercase;
        letter-spacing: 1px;
    }
    .section-break p {
        color: #D5DBDB !important;
        margin-top: 10px !important;
        margin-bottom: 0 !important;
        font-size: 1.1em !important;
    }
</style>

# El Dividendo Educativo en México
### ¿Más años en las aulas se traducen realmente en mayor riqueza estatal?
***

<div class="resumen">
<b>Resumen Ejecutivo:</b> Durante décadas, la política pública ha sostenido que la educación es el principal motor del desarrollo económico. En este reporte, cruzamos 20 años de datos del Censo de Población y Vivienda con el Producto Interno Bruto (PIB) del INEGI para someter esta premisa a una prueba matemática: <b>¿El esfuerzo de mantener a la población en la escuela está generando la riqueza esperada en las diferentes regiones del país?</b>
</div>

<div class="section-break">
    <h2>1. El Punto de Partida</h2>
    <p>Dos décadas de esfuerzo por mantener a México en las aulas</p>
</div>

Construir capital humano es un proceso lento que toma generaciones. No podemos entender el presente económico sin mirar el rezago educativo histórico. 

La siguiente visualización contrasta el Grado Promedio de Escolaridad de cada entidad en el año 2000 frente al 2020. Esta 'Gráfica de Mancuernas' revela la magnitud de la brecha nacional inicial y el esfuerzo de crecimiento de cada estado a lo largo de 20 años.

In [4]:
import plotly.graph_objects as go

# 1. Preparación de datos: Nos quedamos solo con los años extremos
df_dumb = df[['Estado', 'Edu_2000_Total', 'Edu_2020_Total']].copy()

# Ordenamos de mayor a menor según el nivel actual (2020)
df_dumb = df_dumb.sort_values(by='Edu_2020_Total', ascending=True)

# 2. LÓGICA DE VISUALIZACIÓN INSTITUCIONAL (PANORAMA NACIONAL COMxxPLETO)
fig1 = go.Figure()

for i, row in df_dumb.iterrows():
    crecimiento = row['Edu_2020_Total'] - row['Edu_2000_Total']
    
    # Línea de conexión gris
    fig1.add_trace(go.Scatter(
        x=[row['Edu_2000_Total'], row['Edu_2020_Total']],
        y=[row['Estado'], row['Estado']],
        mode='lines',
        line=dict(color='#BDC3C7', width=2), # Gris sólido
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Punto 2000 (Guinda Institucional)
    fig1.add_trace(go.Scatter(
        x=[row['Edu_2000_Total']],
        y=[row['Estado']],
        mode='markers',
        marker=dict(color='#7B241C', size=9),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Punto 2020 (Azul Marino - Aquí ponemos el Tooltip para todos)
    fig1.add_trace(go.Scatter(
        x=[row['Edu_2020_Total']],
        y=[row['Estado']],
        mode='markers',
        marker=dict(color='#002060', size=11),
        showlegend=False,
        customdata=[[row['Edu_2000_Total'], row['Edu_2020_Total'], crecimiento]],
        hovertemplate="<b style='font-size:14px'>%{y}</b><br><br>📚 2020: %{customdata[1]:.1f} años<br>⚪ 2000: %{customdata[0]:.1f} años<br>📈 Crecimiento: +%{customdata[2]:.1f} años<extra></extra>"
    ))

# CONFIGURACIÓN DEL LAYOUT EJECUTIVO
fig1.update_layout(
    title='Contraste del Crecimiento Educativo Nacional: Años de Estudio (2000 vs 2020)',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=850, # Altura perfecta para que los 32 estados se lean claros
    margin=dict(l=20, r=20, t=80, b=40),
    xaxis=dict(
        title='Grado Promedio de Escolaridad (Años)',
        tickmode='linear',
        tick0=6, dtick=1,
        range=[6, 12.5],
        showgrid=True, gridcolor='#EAECEE' # Cuadrícula gris tenue
    ),
    yaxis=dict(
        showgrid=True, gridcolor='#F4F6F7',
        automargin=True # Ajusta el margen para nombres largos
    ),
    font=dict(family="Arial", size=12, color="#333333"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Agregamos las etiquetas de la leyenda manualmente
fig1.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='#7B241C', size=9), name='2000 (Año Base)'))
fig1.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='#002060', size=11), name='2020 (Año Actual)'))

fig1.show()

<div class="section-break">
    <h2>2. Geografía de la Riqueza</h2>
    <p>La distribución del PIB en el territorio nacional</p>
</div>

El desarrollo económico en México no es un fenómeno uniforme; es profundamente regional. Antes de cruzar las variables educativas, es fundamental visualizar cómo se distribuye la generación de riqueza a lo largo del territorio. 

El mapa interactivo muestra el Producto Interno Bruto por entidad federativa en el año 2020. Los tonos oscuros representan los polos de desarrollo industrial, mientras que los tonos claros exponen el rezago.

In [ ]:
import plotly.express as px
import requests
import urllib3

# Apagamos las advertencias rojas de seguridad para que tu libreta luzca limpia
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

try:
    # El truco maestro (verify=False) para evitar bloqueos de red o de sistema operativo
    url = 'https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json'
    respuesta = requests.get(url, verify=False, timeout=15)
    mexico_states = respuesta.json()

    # Si llegó hasta aquí, la descarga fue un éxito. Procedemos con los datos.
    df_map = df.copy()
    df_map['Estado_Mapa'] = df_map['Estado'].replace({'Ciudad de México': 'Distrito Federal'})

    # Construcción del Mapa Institucional
    fig_map = px.choropleth(
        df_map,
        geojson=mexico_states,
        locations='Estado_Mapa',        
        featureidkey='properties.name', 
        color='PIB_2020',            
        color_continuous_scale="Blues", # Azul corporativo
        title='Distribución Geográfica del PIB por Entidad Federativa (2020)',
        labels={'PIB_2020': 'PIB (MDP)'}
    )

    # Ajustes de cámara
    fig_map.update_geos(
        fitbounds="locations", 
        visible=False
    )

    # Estética Ejecutiva
    fig_map.update_layout(
        margin={"r":0,"t":80,"l":0,"b":0},
        height=600,
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family="Arial", size=12, color="#333333"),
        coloraxis_colorbar=dict(
            title="PIB en MDP",
            thickness=15,
            len=0.8
        )
    )

    # Tooltip personalizado
    fig_map.update_traces(
        hovertemplate="<b style='font-size:14px'>%{location}</b><br><br>💰 PIB: $%{z:,.0f} MDP<extra></extra>",
        marker_line_width=0.5, 
        marker_line_color='white'
    )

    fig_map.show()

except Exception as e:
    # Si todo falla, muestra un mensaje formal en lugar de un error de sistema
    print("Aviso: No se pudo cargar el polígono geográfico. Compruebe la conexión a internet de la sala.")

Aviso: No se pudo cargar el polígono geográfico. Compruebe la conexión a internet de la sala.


<div class="section-break">
    <h2>3. El Momento de la Verdad</h2>
    <p>La correlación entre la escuela y el Producto Interno Bruto</p>
</div>

Llegamos a la pregunta central de la investigación. Si la teoría económica es correcta, los estados con mayor escolaridad deberían ser los más ricos. 

En la siguiente gráfica cruzamos ambas realidades. **Si la educación garantiza el éxito económico, los puntos deberían formar una escalera perfecta hacia arriba y a la derecha.** Pase el cursor para descubrir las excepciones nacionales.

In [ ]:
# --- 1. LIMPIEZA RÁPIDA EN MEMORIA ---
# Arreglamos los acentos rotos del INEGI
correcciones = {
    'MÃ©xico': 'México', 'QuerÃ©taro': 'Querétaro', 
    'Nuevo LeÃ³n': 'Nuevo León', 'MichoacÃ¡n de Ocampo': 'Michoacán', 
    'San Luis PotosÃ­': 'San Luis Potosí', 'YucatÃ¡n': 'Yucatán',
    'Ciudad de MÃ©xico': 'Ciudad de México'
}
df['Estado'] = df['Estado'].replace(correcciones, regex=True)

# ELIMINAMOS DUPLICADOS (Aquí se define df_act2)
df_act2 = df.drop_duplicates(subset=['Estado'], keep='first').copy()

# --- 2. DISEÑO PRO: BUBBLE SCATTER PLOT ---
fig2 = px.scatter(
    df_act2,
    x='Edu_2020_Total',
    y='PIB_2020',
    color='PIB_2020', 
    color_continuous_scale='Teal', 
    size='Edu_2020_Total', 
    size_max=18,
    custom_data=['Estado'], # <--- LA LLAVE MAESTRA
    title='La Prueba de Fuego: Educación vs Riqueza por Estado (2020)'
)

# Tooltips (Hover) arreglados para mostrar el estado
fig2.update_traces(
    hovertemplate="<b style='font-size:14px'>%{customdata[0]}</b><br><br>📚 Escolaridad: %{x} años<br>💰 PIB: $%{y:,.0f} MDP<extra></extra>",
    marker=dict(line=dict(width=1.5, color='white'), opacity=0.85)
)

# Estética minimalista
fig2.update_layout(
    plot_bgcolor='white',
    height=600,
    coloraxis_showscale=False, 
    xaxis_title='Años Promedio de Estudio (Eje X)',
    yaxis_title='Producto Interno Bruto en MDP (Eje Y)',
    hovermode='closest',
    margin=dict(l=40, r=40, t=60, b=40)
)
fig2.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F3F4F6', zeroline=False)
fig2.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F3F4F6', zeroline=False)

fig2.show()

<div class="section-break">
    <h2>5. El Impacto Social</h2>
    <p>Asimetría de género en los extremos económicos</p>
</div>

El desarrollo no solo se mide en dinero, sino en equidad social. Nuestra última visualización plantea una interrogante crítica: **¿La riqueza de un estado ayuda a cerrar la brecha educativa estructural?**

Aislamos a las 5 potencias económicas del país frente a los 5 estados con mayor rezago. Utilizando un modelo de 'Gráfica de Mariposa', contrastamos el nivel educativo de hombres (izquierda) y mujeres (derecha).

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

# 1. PREPARACIÓN DE DATOS
df_sorted_pib = df.sort_values(by='PIB_2020', ascending=False)
df_ricos = df_sorted_pib.head(5).copy().sort_values(by='Edu_2020_Total', ascending=True)
df_pobres = df_sorted_pib.tail(5).copy().sort_values(by='Edu_2020_Total', ascending=True)
df_extremos = pd.concat([df_pobres, df_ricos]).reset_index(drop=True)

# --- EL TOQUE MAESTRO: CREAMOS LA ETIQUETA CON EL PIB ---
# Formateamos el PIB con comas y lo unimos al nombre del estado
df_extremos['Estado_Etiqueta'] = df_extremos.apply(
    lambda row: f"{row['Estado']} <span style='color:gray; font-size:10px'>(${row['PIB_2020']:,.0f} MDP)</span>", 
    axis=1
)

# 2. CONSTRUCCIÓN DE LA GRÁFICA DE MARIPOSA
fig5 = make_subplots(
    rows=1, cols=2, 
    shared_yaxes=True, 
    horizontal_spacing=0.02, 
    subplot_titles=('Hombres', 'Mujeres')
)

# PANEL IZQUIERDO: Hombres
fig5.add_trace(go.Bar(
    y=df_extremos['Estado_Etiqueta'], # Usamos la nueva etiqueta con PIB
    x=-df_extremos['Edu_2020_H'], 
    orientation='h',
    name='Hombres',
    marker=dict(color='#002060'),
    text=df_extremos['Edu_2020_H'].round(1),
    textposition='inside',
    insidetextanchor='end',
    textfont=dict(color='white', size=13, family='Arial Black'),
    customdata=df_extremos['PIB_2020'], # Mandamos el PIB crudo al Tooltip
    hovertemplate="<b>%{y}</b><br>💰 PIB 2020: $%{customdata:,.0f} MDP<br>📚 Hombres: %{text} años<extra></extra>"
), row=1, col=1)

# PANEL DERECHO: Mujeres
fig5.add_trace(go.Bar(
    y=df_extremos['Estado_Etiqueta'], # Usamos la nueva etiqueta con PIB
    x=df_extremos['Edu_2020_M'], 
    orientation='h',
    name='Mujeres',
    marker=dict(color='#7B241C'),
    text=df_extremos['Edu_2020_M'].round(1),
    textposition='inside',
    insidetextanchor='start',
    textfont=dict(color='white', size=13, family='Arial Black'),
    customdata=df_extremos['PIB_2020'], # Mandamos el PIB crudo al Tooltip
    hovertemplate="<b>%{y}</b><br>💰 PIB 2020: $%{customdata:,.0f} MDP<br>🔴 Mujeres: %{text} años<extra></extra>"
), row=1, col=2)

# 3. ESTÉTICA Y LÍNEA DIVISORIA
fig5.update_layout(
    title='Contraste Demográfico: Asimetría Educativa (Top 5 Alto PIB vs Top 5 Bajo PIB)',
    plot_bgcolor='white', paper_bgcolor='white',
    height=650, margin=dict(t=100, b=50, l=20, r=20),
    font=dict(family="Arial", size=12, color="#333333"),
    showlegend=False,
    barmode='overlay'
)

# --- EL DIVISOR ---
fig5.add_hline(
    y=4.5, 
    line_width=2, 
    line_dash="dash", 
    line_color="#7F8C8D"
)

fig5.add_annotation(
    xref="paper", yref="y",
    x=0.5, y=4.5, 
    text="↑ ALTO PIB | BAJO PIB ↓",
    showarrow=False,
    font=dict(size=11, color="#7F8C8D", weight="bold"),
    bgcolor="white", 
    xanchor="center",
    yanchor="middle"
)

# Ajustes de Ejes X 
fig5.update_xaxes(
    tickvals=[-12, -10, -8, -6, -4, -2, 0], 
    ticktext=['12', '10', '8', '6', '4', '2', '0'], 
    range=[-12.5, 0], title='', row=1, col=1, showgrid=True, gridcolor='#F2F2F2'
)
fig5.update_xaxes(
    tickvals=[0, 2, 4, 6, 8, 10, 12],
    range=[0, 12.5], title='', row=1, col=2, showgrid=True, gridcolor='#F2F2F2'
)

# Ajustes del Eje Y (Permitir HTML en los nombres)
fig5.update_yaxes(showgrid=False, tickfont=dict(size=12, color='#333333', weight='bold'))

# Coloreado de Títulos
fig5.layout.annotations[0].update(font=dict(size=16, color='#002060', weight='bold'))
fig5.layout.annotations[1].update(font=dict(size=16, color='#7B241C', weight='bold'))

fig5.show()

<div class="section-break">
    <h2>4. Matriz Estratégica</h2>
    <p>El Retorno de Inversión (ROI) Educativo en la última década</p>
</div>

Conocer la posición actual es útil, pero necesitamos entender la **dinámica de crecimiento**. ¿Los estados que más aumentaron su nivel educativo en la última década (2010-2020) lograron capitalizarlo en crecimiento económico?

Dividimos el panorama nacional en cuatro cuadrantes cruzando el porcentaje de crecimiento del PIB frente al incremento neto en años de escolaridad.

Para responder a esto, calculamos el porcentaje de crecimiento del PIB frente al incremento neto en años de escolaridad, dividiendo el panorama nacional en cuatro cuadrantes de desempeño:
* **Líderes de Crecimiento (Arriba-Derecha):** Alto esfuerzo educativo con alta recompensa económica.
* **Rezago Estructural (Abajo-Izquierda):** Entidades atrapadas en bajo crecimiento educativo y económico.
* **Crecimiento Atípico (Arriba-Izquierda):** Crecimiento económico impulsado por factores ajenos a la educación (ej. industria extractiva o manufactura base).
* **Esfuerzo sin Retorno (Abajo-Derecha):** Estados que mejoraron su educación, pero cuya economía no logró capitalizar ese talento.

In [ ]:
import plotly.express as px

# 1. INGENIERÍA DE CARACTERÍSTICAS (Cálculos de crecimiento)
# Calculamos cuánto creció la educación (absoluto) y el PIB (porcentual) en la década 2010-2020
df['Delta_Edu_10_20'] = df['Edu_2020_Total'] - df['Edu_2010_Total']
df['Delta_PIB_pct'] = ((df['PIB_2020'] - df['PIB_2010']) / df['PIB_2010']) * 100

# 2. DEFINICIÓN DE LOS 4 CUADRANTES (Basado en medianas nacionales)
med_edu = df['Delta_Edu_10_20'].median()
med_pib = df['Delta_PIB_pct'].median()

def clasificar_cuadrante(row):
    if row['Delta_Edu_10_20'] >= med_edu and row['Delta_PIB_pct'] >= med_pib:
        return 'Líderes de Crecimiento'
    elif row['Delta_Edu_10_20'] < med_edu and row['Delta_PIB_pct'] < med_pib:
        return 'Rezago Estructural'
    elif row['Delta_Edu_10_20'] >= med_edu and row['Delta_PIB_pct'] < med_pib:
        return 'Esfuerzo sin Retorno'
    else:
        return 'Crecimiento Atípico'

df['Cuadrante'] = df.apply(clasificar_cuadrante, axis=1)

# 3. CONSTRUCCIÓN DE LA MATRIZ ESTRATÉGICA
fig4 = px.scatter(
    df, 
    x='Delta_Edu_10_20', 
    y='Delta_PIB_pct', 
    color='Cuadrante',
    size='PIB_2020', # El tamaño de la burbuja refleja su economía actual
    size_max=22, 
    custom_data=['Estado', 'Cuadrante'],
    color_discrete_map={
        'Líderes de Crecimiento': '#002060',  # Azul Institucional
        'Rezago Estructural': '#7B241C',      # Guinda
        'Esfuerzo sin Retorno': '#5D6D7E',    # Gris Pizarra
        'Crecimiento Atípico': '#D4AC0D'      # Dorado
    },
    title='Matriz de ROI Educativo: Esfuerzo Escolar vs Crecimiento Económico (2010-2020)'
)

# 4. ESTÉTICA INSTITUCIONAL Y LÍNEAS DIVISORIAS
fig4.update_layout(
    plot_bgcolor='white', 
    height=650,
    margin=dict(t=80, b=50),
    xaxis_title='Esfuerzo Educativo (Años de estudio ganados)',
    yaxis_title='Retorno Económico (Crecimiento del PIB en %)',
    legend_title_text='Clasificación Estratégica',
    font=dict(family="Arial", size=12, color="#333333")
)

# Trazamos la cruz de medianas para dividir visualmente los cuadrantes
fig4.add_vline(x=med_edu, line_width=2, line_dash="dash", line_color="#BDC3C7")
fig4.add_hline(y=med_pib, line_width=2, line_dash="dash", line_color="#BDC3C7")

fig4.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F2F2F2', zeroline=False)
fig4.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F2F2F2', zeroline=False)

# Tooltip limpio y directo
fig4.update_traces(
    hovertemplate="<b style='font-size:14px'>%{customdata[0]}</b><br>Clasificación: %{customdata[1]}<br><br>📈 Avance Educativo: +%{x:.2f} años<br>💰 Crecimiento PIB: +%{y:.1f}%<extra></extra>",
    marker=dict(line=dict(width=1, color='white'), opacity=0.85)
)

fig4.show()

<div class="section-break" style="background-color: #7B241C; border-left: 10px solid #002060;">
    <h2>Conclusión</h2>
    <p>El veredicto de los datos procesados</p>
</div>

Los datos no mienten. A lo largo de dos décadas, México ha logrado mantener a más personas en las aulas. Sin embargo, nuestra investigación revela un panorama complejo: **la educación por sí sola no es una garantía automática de riqueza estatal**, pero la falta de ella sí es una barrera económica casi imposible de superar.

Más revelador aún es nuestro último hallazgo: el desarrollo económico en los estados con mayor PIB impulsa un ecosistema social más equitativo, logrando la simetría entre géneros. Un dividendo educativo que va mucho más allá del dinero.